In [5]:
import pandas as pd
from pathlib import Path

# Path 
path = Path("../data/raw/text_files/1993/legislatives")
documents = []

# To get all .txt files in the folder and read their content
for file in path.glob("*.txt"):
    try:
        texte = file.read_text(encoding='utf-8', errors='ignore')
        documents.append({
            "filename": file.name,
            "texte_brut": texte})
    except Exception as e:
        print(f"Error with {file.name} : {e}")


df = pd.DataFrame(documents)
print(f"{len(df)} documents in the dataframe")

5936 documents in the dataframe


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.corpus import stopwords

# Téléchargement des mots vides (stopwords) si ce n'est pas déjà fait
nltk.download('stopwords', quiet=True)
french_stopwords = stopwords.words('french')

# Ajout manuel de mots parasites souvent liés à l'OCR ou aux manifestos
mots_parasites = ['plus', 'tout', 'faire', 'cette', 'comme', 'être', 'avoir', 'candidat', 'élection']
mes_stopwords = french_stopwords + mots_parasites

# Paramétrage du vectoriseur
vectorizer = TfidfVectorizer(
    stop_words=mes_stopwords,
    max_df=0.75,     # Ignore les mots présents dans plus de 75% des textes (trop communs)
    min_df=5,        # Ignore les mots présents dans moins de 5 textes (fautes de frappe OCR)
    max_features=5000 # Ne garde que les 5000 mots les plus pertinents du vocabulaire
)

# Application sur tes textes
X_tfidf = vectorizer.fit_transform(df['texte_brut'])
print(f"Matrice créée : {X_tfidf.shape[0]} documents, {X_tfidf.shape[1]} mots de vocabulaire.")

Matrice créée : 5936 documents, 5000 mots de vocabulaire.


In [10]:
from sklearn.decomposition import NMF
import numpy as np

nombre_de_themes = 15

# Création et entraînement du modèle
nmf_model = NMF(n_components=nombre_de_themes, random_state=42)
W = nmf_model.fit_transform(X_tfidf) # W contient le score de chaque thème pour chaque document

# Affichage des mots les plus importants pour chaque thème
mots_du_vocabulaire = vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(nmf_model.components_):
    print(f"\nThème {topic_idx + 1} :")
    # On récupère les indices des 10 mots avec le plus fort score
    top_mots_idx = topic.argsort()[:-11:-1]
    top_mots = [mots_du_vocabulaire[i] for i in top_mots_idx]
    print(" | ".join(top_mots))


Thème 1 :
français | udf | rpr | immigrés | immigration | voulez | carte | front | supprimant | sauver

Thème 2 :
animaux | nature | nouveaux | rassemblement | existence | marseille | activités | ecologistes | doivent | homme

Thème 3 :
citoyens | faut | développement | gauche | politique | doit | sociale | sans | emploi | vie

Thème 4 :
patrons | payer | patronat | maintenir | travailleurs | leurs | devons | bourgeoisie | bénéfices | vendre

Thème 5 :
écologie | entente | ecologistes | verts | ecologie | generation | chance | écologistes | peu | vie

Thème 6 :
die | der | und | für | zu | in | den | sie | das | von

Thème 7 :
naturelle | loi | cohérence | connaissance | programme | fondée | maharishi | védique | criminalité | éliminer

Thème 8 :
communiste | droite | vote | parti | neuf | dirigeants | politique | actuelle | autre | progrès

Thème 9 :
12 | refus | ans | instauration | maintien | mise | humiliation | mixte | assez | place

Thème 10 :
travailleurs | parti | unité | solu

In [9]:
# On cherche l'index du thème qui a le score maximum pour chaque document
df['theme_dominant'] = np.argmax(W, axis=1)

# On peut voir le résultat
print(df[['filename', 'theme_dominant']].head(10))

                             filename  theme_dominant
0  EL190_L_1993_03_016_02_2_PF_02.txt               2
1  EL189_L_1993_03_003_02_1_PF_01.txt               4
2  EL193_L_1993_03_063_01_2_PF_01.txt               2
3  EL196_L_1993_03_076_12_1_PF_08.txt               1
4  EL190_L_1993_03_030_05_2_PF_01.txt               2
5  EL192_L_1993_03_051_04_2_PF_01.txt               2
6  EL192_L_1993_03_044_04_2_PF_01.txt               2
7  EL190_L_1993_03_022_02_1_PF_06.txt               1
8  EL198_L_1993_03_092_09_1_PF_03.txt               2
9  EL192_L_1993_03_056_03_1_PF_06.txt               1


In [ ]:
import spacy
import pandas as pd

# 1. Chargement du modèle linguistique français
# On désactive les fonctions avancées (ner, parser) pour accélérer le traitement
nlp = spacy.load("fr_core_news_sm", disable=["ner", "parser"])

# 2. Ajout de mots parasites spécifiques à ton corpus politique
# Ces mots n'apportent aucune information économique et pollueraient tes thèmes
mots_parasites = {"candidat", "élection", "république", "français", "madame", "monsieur", "plus", "tout", "faire", "dire"}

# 3. Création de la fonction de nettoyage
def obtenir_mots_propres(texte):
    # Sécurité : on s'assure que la cellule contient bien du texte
    if not isinstance(texte, str):
        return []
        
    # Le texte est converti en minuscules et analysé par l'intelligence de spaCy
    doc = nlp(texte.lower())
    
    mots_propres = []
    
    # On boucle sur chaque mot (token) du document
    for token in doc:
        # On rejette la ponctuation, les espaces, les nombres et les "stop words" classiques
        if not token.is_punct and not token.is_space and not token.like_num and not token.is_stop:
            
            # On extrait le lemme (la racine du mot dans le dictionnaire)
            lemme = token.lemma_
            
            # On vérifie que le lemme n'est pas dans nos mots parasites et fait plus de 2 lettres
            if lemme not in mots_parasites and len(lemme) > 2:
                mots_propres.append(lemme)
                
    return mots_propres

print("Début du nettoyage des textes. Cette opération peut prendre quelques minutes...")

# 4. Application de la fonction sur la colonne de textes bruts
# Cela va créer une nouvelle colonne contenant uniquement des listes de mots épurés
df['mots_propres'] = df['texte_brut'].apply(obtenir_mots_propres)

print("Nettoyage terminé ! Voici un aperçu des mots propres :")
print(df[['texte_brut', 'mots_propres']].head())

Début du nettoyage des textes. Cette opération peut prendre quelques minutes...
Nettoyage terminé ! Voici un aperçu des mots propres :
                                          texte_brut  \
0  Sciences Po / fonds CEVIPOF\nRépublique França...   
1  ENTENTE DES ECOLOGISTES\nÉlections législative...   
2  DÉPARTEMENT DU PUY-DE-DÔME\nRÉPUBLIQUE FRANÇAI...   
3  RÉPUBLIQUE FRANÇAISE - ÉLECTIONS LÉGISLATIVES ...   
4  DANILET ALAIN\nSUPPLEANT : Christian BURGLÉ (P...   

                                        mots_propres  
0  [science, fonds, cevipof, département, charent...  
1  [entente, ecologiste, législatif, scrutin, mar...  
2  [département, puy-de-dôme, législatif, mars, 1...  
3  [législatif, mars, science, fonds, cevipof, su...  
4  [danilet, alain, suppleant, christian, burgler...  


In [20]:
import numpy as np
from gensim.corpora import Dictionary
from gensim.models import LdaModel



textes_tokenises = df['mots_propres'].tolist() 

# Création du dictionnaire (le vocabulaire) et du corpus (la matrice d'occurrences)
dictionnaire = Dictionary(textes_tokenises)

# Optionnel mais recommandé : filtrer les mots trop rares ou trop fréquents
dictionnaire.filter_extremes(no_below=5, no_above=0.75)
corpus = [dictionnaire.doc2bow(texte) for texte in textes_tokenises]

# Étape 2 : Création de la matrice Eta (les a priori)
nombre_de_themes = 15
taille_vocabulaire = len(dictionnaire)

# On initialise la matrice avec une valeur de base très faible pour laisser le modèle libre
eta_matrix = np.full((nombre_de_themes, taille_vocabulaire), 0.01)

# Étape 3 : Forcer le Thème 0 sur le chômage et l'économie
mots_graines = ['chômage', 'licenciement', 'chômeur']

# On attribue un poids massif à ces mots uniquement pour la ligne 0 (le premier thème)
for mot in mots_graines:
    # On vérifie d'abord que le mot a survécu au filtrage du vocabulaire
    if mot in dictionnaire.token2id:
        mot_id = dictionnaire.token2id[mot]
        # On multiplie le poids par 1000 pour forcer l'algorithme à les regrouper ici
        eta_matrix[0, mot_id] = 100 

print("Matrice Eta configurée. Lancement de l'entraînement...")

# Étape 4 : Entraînement du modèle LDA avec notre matrice personnalisée
lda_guide = LdaModel(
    corpus=corpus,
    id2word=dictionnaire,
    num_topics=nombre_de_themes,
    eta=eta_matrix, # C'est ici que la magie opère
    random_state=42,
    passes=10 # Nombre de passages sur l'ensemble du corpus
)

# Étape 5 : Affichage des résultats
print("\nLes thèmes extraits par le modèle :")
for idx, topic in lda_guide.print_topics(num_words=10):
    print(f"Thème {idx} : {topic}\n")

Matrice Eta configurée. Lancement de l'entraînement...

Les thèmes extraits par le modèle :
Thème 0 : 0.040*"nature" + 0.035*"animal" + 0.032*"nouveau" + 0.024*"rassemblement" + 0.016*"nom" + 0.016*"bulletin" + 0.016*"homme" + 0.016*"existence" + 0.016*"ecologiste" + 0.016*"activité"

Thème 1 : 0.017*"loi" + 0.014*"naturel" + 0.013*"national" + 0.010*"programme" + 0.009*"création" + 0.008*"france" + 0.007*"parti" + 0.007*"etat" + 0.006*"nature" + 0.006*"chômage"

Thème 2 : 0.033*"die" + 0.022*"und" + 0.021*"das" + 0.019*"der" + 0.012*"alsace" + 0.012*"durch" + 0.012*"den" + 0.012*"wir" + 0.011*"pari" + 0.010*"eine"

Thème 3 : 0.016*"maire" + 0.015*"député" + 0.011*"conseiller" + 0.011*"tour" + 0.010*"national" + 0.009*"électeur" + 0.009*"confiance" + 0.008*"dimanche" + 0.008*"général" + 0.008*"voter"

Thème 4 : 0.031*"écologie" + 0.028*"ecologiste" + 0.027*"entente" + 0.024*"écologiste" + 0.024*"vert" + 0.018*"environnement" + 0.017*"vie" + 0.017*"ecologie" + 0.014*"nouveau" + 0.014*"a